# 🧠 PyTorch Tabular Deep Learning (MLP with Entity Embeddings)

- **Thành viên:** C (Phạm Hồng Nam)
- **Mã thí nghiệm:** `mlp_c_v1`
- **Họ mô hình:** **Deep Learning (Học sâu thuần túy)**, 100% không dùng LLM
- **Kiến trúc:**
  - Categorical Features: `nn.Embedding(num_categories, emb_dim)`
  - Continuous Features: `StandardScaler` + `BatchNorm1d`
  - Dense Backbone: Linear(256) -> SiLU -> Dropout(0.3) -> Linear(128) -> SiLU -> Dropout(0.2) -> Linear(64) -> SiLU -> Linear(1)
  - Optimizer: `AdamW` (lr=3e-3, CosineAnnealingLR)
- **Kết quả:** CV AUC = **0.94167**, Public LB = **0.94326**

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "C" else Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "C" / "outputs"
SUB_DIR = ROOT / "submissions"
EXP_ID = "mlp_c_v1"
SEED = 42
N_SPLITS = 5
EPOCHS = 6
BATCH_SIZE = 4096

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
folds_df = pd.read_csv(DATA_DIR / "folds.csv")
sub_df = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_df["fold"] = folds_df["fold"]

cat_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
bool_cols = ["is_foreign_transaction", "is_new_merchant", "used_vpn", "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]
num_cols = [c for c in train_df.columns if c not in cat_cols + bool_cols + ["id", "is_fraud", "fold"]]

for c in bool_cols:
    train_df[c] = train_df[c].astype(float)
    test_df[c] = test_df[c].astype(float)

oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
train_df[cat_cols] = oe.fit_transform(train_df[cat_cols]).astype(int)
test_df[cat_cols] = oe.transform(test_df[cat_cols]).astype(int)

cont_cols = num_cols + bool_cols
scaler = StandardScaler()
train_df[cont_cols] = scaler.fit_transform(train_df[cont_cols])
test_df[cont_cols] = scaler.transform(test_df[cont_cols])

cat_cardinalities = [train_df[c].nunique() for c in cat_cols]
emb_dims = [(num_cat, min(16, (num_cat + 1) // 2 + 2)) for num_cat in cat_cardinalities]
print("Embedding Dims:", emb_dims)


In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, emb_dims, num_cont):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(num_cat, dim) for num_cat, dim in emb_dims])
        total_emb_dim = sum(dim for _, dim in emb_dims)
        in_dim = total_emb_dim + num_cont
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x_cat, x_cont):
        emb_outs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(emb_outs + [x_cont], dim=1)
        return self.mlp(x).squeeze(1)

model = TabularMLP(emb_dims, len(cont_cols)).to(device)
print(model)


In [ ]:
# Huấn luyện 5-Fold Stratified
oof_preds = np.load(OUTPUT_DIR / f"{EXP_ID}_oof.npy")
test_preds = np.load(OUTPUT_DIR / f"{EXP_ID}_test.npy")
overall_auc = roc_auc_score(train_df["is_fraud"], oof_preds)
print("Overall OOF AUC:", round(overall_auc, 5))
